# Conscious Access

**Author:** Teo Fantacci  
**Purpose:** Pedagogical companion to Froudist-Walsh et al. (in press, *Biol Psychiatry*) — illustrates a 2-area firing-rate model of cortical ignition as a substrate for conscious access, following Klatzmann et al. (2025). Corresponds to **Fig 3C** and **§3.2** of the review.

## How to cite
If you find this code useful for your research, please cite:
- The original paper: Klatzmann U, Froudist-Walsh S, Bliss DP, Theodoni P, Mejías J, Niu M, et al. (2025): A dynamic bifurcation mechanism explains cortex-wide neural correlates of conscious access. *Cell Rep* 44. https://doi.org/10.1016/j.celrep.2025.115372
- The review: Froudist-Walsh S, Ivanov TG, Magrou L, Cho H, Busch AN, Muller LE, et al. (in press): Mechanistic computational models of cognition across scales and species. *Biol Psychiatry*.

## Requirements
```bash
pip install numpy matplotlib
```
Runs in Google Colab (recommended) or locally with Jupyter (Python ≥ 3.8).  
No external data files required — all simulations are self-contained.

## Notes
All code is original. No third-party data or figures are included.
Simulation outputs are generated at runtime and not stored in the repository.


## Model overview

This notebook implements a **2-area firing-rate model of cortical ignition**: a sensory area (Se, Si) and a frontoparietal area (FPe, FPi), connected by feedforward (AMPA-dominant) and feedback (mixed AMPA + NMDA) pathways. A brief sensory stimulus can trigger an "all-or-none" ignition of the frontoparietal network — the model's signature of conscious access.

## Dynamics

**Notation:** populations are indexed $k \in \{\text{Se}, \text{Si}, \text{FPe}, \text{FPi}\}$. The two excitatory pools are $e \in \{\text{Se}, \text{FPe}\}$ (rates from $r_E$ below); the two inhibitory pools are $\{\text{Si}, \text{FPi}\}$ (rates from $r_I$). $r$ denotes firing rates (Hz), $S$ synaptic gating fractions, $J$ connectivity weights, $\tau$ time constants.

All populations carry AMPA and GABA gating; only the excitatory pools carry NMDA gating:

$$
\frac{dS^{\text{NMDA}}_e}{dt} = -\frac{S^{\text{NMDA}}_e}{\tau_{\text{NMDA}}} + \gamma\,(1 - S^{\text{NMDA}}_e)\, r_e
\qquad (e \in \{\text{Se}, \text{FPe}\})
$$

$$
\frac{dS^{\text{AMPA}}_k}{dt} = -\frac{S^{\text{AMPA}}_k}{\tau_{\text{AMPA}}} + r_k
\qquad
\frac{dS^{\text{GABA}}_k}{dt} = -\frac{S^{\text{GABA}}_k}{\tau_{\text{GABA}}} + r_k
$$

Total synaptic current to population $k$ (the GABA weights $J^{\text{GABA}}$ are signed negative for inhibition):

$$
I_k = I_k^{\text{back}} + \sum_j J^{\text{AMPA}}_{kj}\, S^{\text{AMPA}}_j + \sum_j J^{\text{NMDA}}_{kj}\, S^{\text{NMDA}}_j + \sum_j J^{\text{GABA}}_{kj}\, S^{\text{GABA}}_j + I_k^{\text{ext}}(t) + \eta_k(t)
$$

Local recurrent self-excitation on each excitatory pool is scaled by a hierarchy factor (sensory areas excitable less than FP areas, $h_{\text{sensory}} < h_{\text{frontoparietal}}$):

$$
J^{\text{AMPA}}_{ee} = h_e \cdot g^{\text{AMPA}}_{\text{self}}
\qquad
J^{\text{NMDA}}_{ee} = h_e \cdot g^{\text{NMDA}}_{\text{self}}
$$

Firing rates from the f-I curves (Abbott–Chance for E, linear-threshold for I; same as WTA/WM):

$$
r_E = \frac{aI - b}{1 - \exp\bigl(-d\,(aI - b)\bigr)}
\qquad
r_I = \max\bigl(c\, I - r_0,\, 0\bigr)
$$

The hierarchy gradient $h_e$ is what makes the sensory area **monostable** (it cannot sustain its own activity) but the frontoparietal area **bistable** (slow NMDA-driven self-excitation supports a high-rate attractor). A brief stimulus to Se either dies out (miss) or — when stimulus plus noise push the system past the unstable separatrix — ignites the FP attractor (hit). Feedforward Se→FPe is AMPA-dominant (fast propagation); feedback FPe→Se and FPe→Si is mixed AMPA+NMDA (slower, stabilizing).

## Reference and scope

The model is **inspired by** Klatzmann et al. (2025), *Cell Reports* 44:115372, which presents a 40-area connectome-based model of conscious access. We build a **2-area reduction** of that model that preserves its key mechanism: the asymmetric **AMPA/NMDA gradient** and the **subnetwork bistability** regime (sensory monostable, FP bistable).

What we keep from Klatzmann 2025:
- **Local circuit per area**: one excitatory pool (E) and one inhibitory pool (I). Klatzmann actually has E1, E2, I per area (for selectivity); we simplify to a single E per area since we are not modeling stimulus-feature decisions here.
- **Inter-areal connectivity pattern** (p. 6 of Klatzmann):
  - Feedforward (Se → FPe): **AMPA-dominant**, targets E
  - Feedback (FPe → Se, FPe → Si): **mixed AMPA + NMDA**, targets both E and I, with greater NMDA contribution
- **Hierarchical AMPA/NMDA gradient** (p. 7): scales local recurrent excitation by `h0` (sensory) < `h1` (frontoparietal). This is a simplification of Klatzmann's spine-count gradient `z_E`.
- **OU noise** on all populations (Klatzmann eq. 12).

What we drop / simplify:
- The 40-area cortex-wide architecture → reduced to 2 areas.
- The dendritic compartment function `F` (eq. 17) → omitted; long-range currents enter the soma directly. This is a substantial simplification: F clips long-range currents to [0, 300 pA] which prevents runaway. Our 2-area system is small enough that we can stay in stable regimes without it.
- E1/E2 within-area selectivity → we use one E per area.
- f-I curve: we keep the W&W 2006 / WTA / WM values (`a=310, b=125, d=0.16` for E; `c=615, r₀=177` for I) for **consistency across notebooks**. Klatzmann uses different values (`a=0.135 Hz/pA, b=54 Hz, d=0.308 s` for E); the dynamics would be similar after re-tuning weights.

## Target dynamics

The "subnetwork bistability" regime of Klatzmann (Fig. 5 of the paper). After tuning the weights, the model produces:

- **Sensory area (Se)**: ~0.3 Hz in the delay period (transient response only during stimulus).
- **Sensory inhibition (Si)**: ~46 Hz — same order of magnitude as the FP rates due to the steep I f-I curve. (Originally targeted "just above Se" but this is in tension with the W&W-style I f-I curve we keep for cross-notebook consistency; see note below.)
- **Frontoparietal (FPe)**: ~24 Hz — within the 20–40 Hz target range.
- **Frontoparietal inhibition (FPi)**: ~34 Hz — above FPe, balancing it.

**Note on Si firing rate**: The W&W 2006 inhibitory f-I curve (`r = max(615·I − 177, 0)`) is steep, so any I population with non-trivial drive ends up at 20+ Hz. Getting Si much closer to Se (e.g., < 10 Hz) would require a shallower I f-I curve, which would deviate from the WTA and WM notebooks. We accepted this trade-off for cross-notebook consistency.

## Noise

Two options, both applied to **all 4 populations** (E and I), matching Klatzmann (eq. 12):
- **`"ou"`**: Ornstein–Uhlenbeck process filtered by τ_AMPA = 2 ms (default; Klatzmann's choice).
- **`"gaussian"`**: iid Gaussian per timestep (simpler; effective amplitude depends on `dt`).


## Setup: Drive and results folder (Colab) / local fallback


In [ ]:
import os

# If running in Google Colab, optionally point this to your own Drive folder.
# Otherwise leave as './' — the notebook will work in the current directory.
tutorial_path = './'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Optional: uncomment and edit the line below to save to your Drive instead
    # tutorial_path = '/content/drive/MyDrive/your_folder_here/'
    print(f"Running in Colab. Working in: {tutorial_path}")
except (ImportError, ModuleNotFoundError):
    tutorial_path = os.getcwd()
    print(f"Running locally. Working in: {tutorial_path}")

saved_results = os.path.join(tutorial_path, 'saved_results')
os.makedirs(saved_results, exist_ok=True)
print(f"Results will be saved under: {saved_results}")


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt


## Parameters

Slim dict exposing only the parameters most likely to be varied. Hard-coded inside the simulation: `dt`, `tau_*`, `gamma_NMDA`, background currents `I_back`, f-I curve constants, noise amplitudes, `S_init`. See the `run_simulation` function body for these.

**Indices**: `0 = Se` (sensory excitatory), `1 = Si` (sensory inhibitory), `2 = FPe` (frontoparietal excitatory), `3 = FPi` (frontoparietal inhibitory).


## Building the connectivity matrices

Three `[4, 4]` matrices: `J_NMDA`, `J_AMPA`, `J_GABA`. All use the convention `J[TO, FROM]` with the current computed as `I = J @ S`. This matches the WTA and WM notebooks.

The hierarchy scaling `h0` (sensory) and `h1` (frontoparietal) is applied to the local recurrent self-excitation weights only (Klatzmann's spine-count gradient simplification).


In [ ]:
def build_J_matrices(conn, hierarchy):
    """Assemble J_AMPA, J_NMDA, J_GABA for [Se, Si, FPe, FPi].

    Indexing convention: J[TO, FROM]. Current = J @ S (no transpose).
    GABA entries enter with a negative sign here; the input weights are
    stored as positive magnitudes in the dict for clarity.
    """
    h0 = hierarchy["h_sensory"]
    h1 = hierarchy["h_frontoparietal"]

    J_AMPA = np.zeros((4, 4))
    J_NMDA = np.zeros((4, 4))
    J_GABA = np.zeros((4, 4))

    # ---- Local recurrent self-excitation (E -> same E), with hierarchy ----
    # Se (index 0): scaled by h0
    J_AMPA[0, 0] = conn["g_AMPA_self"] * h0
    J_NMDA[0, 0] = conn["g_NMDA_self"] * h0
    # FPe (index 2): scaled by h1
    J_AMPA[2, 2] = conn["g_AMPA_self"] * h1
    J_NMDA[2, 2] = conn["g_NMDA_self"] * h1

    # ---- Feedforward Se -> FPe (AMPA-dominant) ----
    # TO=FPe(2), FROM=Se(0)
    J_AMPA[2, 0] = conn["g_AMPA_Se_to_FPe"]

    # ---- Feedback FPe -> Se (mixed AMPA + NMDA) ----
    # TO=Se(0), FROM=FPe(2)
    J_AMPA[0, 2] = conn["g_AMPA_FPe_to_Se"]
    J_NMDA[0, 2] = conn["g_NMDA_FPe_to_Se"]

    # ---- Feedback FPe -> Si (mixed AMPA + NMDA, NMDA larger) ----
    # TO=Si(1), FROM=FPe(2)
    J_AMPA[1, 2] = conn["g_AMPA_FPe_to_Si"]
    J_NMDA[1, 2] = conn["g_NMDA_FPe_to_Si"]

    # ---- Local E -> I within each area ----
    # Se -> Si:  TO=1, FROM=0
    J_AMPA[1, 0] = conn["g_AMPA_Se_to_Si"]
    # FPe -> FPi: TO=3, FROM=2
    J_AMPA[3, 2] = conn["g_AMPA_FPe_to_FPi"]

    # ---- Local I -> E within each area (negative sign for inhibition) ----
    # Si -> Se: TO=0, FROM=1
    J_GABA[0, 1] = -conn["g_GABA_Si_to_Se"]
    # FPi -> FPe: TO=2, FROM=3
    J_GABA[2, 3] = -conn["g_GABA_FPi_to_FPe"]

    # ---- Local I -> I self-inhibition (negative) ----
    J_GABA[1, 1] = -conn["g_GABA_Si"]
    J_GABA[3, 3] = -conn["g_GABA_FPi"]

    return J_AMPA, J_NMDA, J_GABA


## f-I (input → firing rate) curves

Same as WTA and WM: Abbott–Chance for E, linear-threshold for I. We use the W&W 2006 values for cross-notebook consistency (see overview for caveat).


In [ ]:
def fI_excitatory(I):
    """Abbott-Chance input-output function for E populations."""
    a, b, d = 310.0, 125.0, 0.16
    x = a * I - b
    return x / (1.0 - np.exp(-d * x) + 1e-6)


def fI_inhibitory(I):
    """Linear-threshold rate for I populations."""
    c, r_0 = 615.0, 177.0
    return np.maximum(c * I - r_0, 0.0)


## Core simulation

Hard-coded constants:

- `dt = 0.5 ms`
- `tau_AMPA = 2 ms`, `tau_NMDA = 100 ms`, `tau_GABA = 5 ms`, `tau_OU = tau_AMPA = 2 ms`
- `gamma_NMDA = 0.641` (E populations only; I has γ=1 with no saturation)
- `I_back = [0.360, 0.328, 0.340, 0.328] nA` for [Se, Si, FPe, FPi]
- `sigma_gauss = 0.002 nA`, `sigma_ou = 0.002 nA` (applied to all 4 populations)
- `S_init = 0` (clean start, no initial-condition transient)


In [ ]:
def run_simulation(params, seed=None):
    """Run one trial of the 2-area ignition network.

    Returns
    -------
    dict with keys: 'time', 'R' (4 x n_steps), 'S_AMPA', 'S_NMDA', 'S_GABA',
    'stim_on', 'stim_off', 'params'.
    """
    # --- Hard-coded constants ---
    dt = 0.0005
    tau_AMPA = 0.002
    tau_NMDA = 0.100
    tau_GABA = 0.005
    tau_OU   = tau_AMPA
    gamma_NMDA = 0.641
    I_back = np.array([0.360, 0.328, 0.340, 0.328])  # [Se, Si, FPe, FPi]
    sigma_gauss = 0.002
    sigma_ou    = 0.002
    S_init = 0.0

    # --- Unpack ---
    p_t = params["time"]
    noise_type = params["noise_type"]
    if seed is None:
        seed = params["simulation"].get("seed", None)
    rng = np.random.default_rng(seed)

    # --- Time grid ---
    time = np.arange(0.0, p_t["trial_length"], dt)
    n_steps = len(time)
    stim_on, stim_off = p_t["stim_on"], p_t["stim_off"]
    stim_strength = params["stim_strength"]

    # --- Connectivity ---
    J_AMPA, J_NMDA, J_GABA = build_J_matrices(
        params["connectivity"], params["hierarchy"]
    )

    # --- State arrays ---
    S_AMPA = np.zeros((4, n_steps))
    S_NMDA = np.zeros((4, n_steps))
    S_GABA = np.zeros((4, n_steps))
    R      = np.zeros((4, n_steps))
    S_AMPA[:, 0] = S_init
    S_NMDA[:, 0] = S_init
    S_GABA[:, 0] = S_init

    I_noise = np.zeros(4)        # OU noise state (one per population)

    # E and I indices
    E_idx = [0, 2]   # Se, FPe
    I_idx = [1, 3]   # Si, FPi

    # --- Euler integration ---
    for t in range(n_steps - 1):
        # External stimulus to Se only
        I_ext = np.zeros(4)
        if stim_on <= time[t] <= stim_off:
            I_ext[0] = stim_strength

        # --- Noise (applied to all 4 populations) ---
        if noise_type == "gaussian":
            noise_vec = rng.standard_normal(4) * sigma_gauss
        elif noise_type == "ou":
            xi = rng.standard_normal(4)
            I_noise = (I_noise
                       - (I_noise / tau_OU) * dt
                       + sigma_ou * np.sqrt(dt / tau_OU) * xi)
            noise_vec = I_noise.copy()
        else:
            raise ValueError(f"Unknown noise_type: {noise_type!r}. Use 'gaussian' or 'ou'.")

        # --- Total synaptic current per population ---
        # J[TO, FROM] @ S gives the input to each TO population.
        I_tot = (I_back
                 + J_AMPA @ S_AMPA[:, t]
                 + J_NMDA @ S_NMDA[:, t]
                 + J_GABA @ S_GABA[:, t]
                 + I_ext
                 + noise_vec)

        # --- Firing rates ---
        R[E_idx, t] = fI_excitatory(I_tot[E_idx])
        R[I_idx, t] = fI_inhibitory(I_tot[I_idx])

        # --- Gating dynamics ---
        # AMPA (all populations, fast, no saturation)
        dS_AMPA = -S_AMPA[:, t] / tau_AMPA + R[:, t]
        # GABA (all populations, fast, no saturation)
        dS_GABA = -S_GABA[:, t] / tau_GABA + R[:, t]
        # NMDA: only E populations have NMDA gating in this model.
        # I populations have no NMDA-receptor inputs (J_NMDA columns 1, 3 are zero),
        # so we don't track S_NMDA on them. S_NMDA[I_idx] stays at 0.
        dS_NMDA = np.zeros(4)
        dS_NMDA[E_idx] = (-S_NMDA[E_idx, t] / tau_NMDA
                          + gamma_NMDA * (1.0 - S_NMDA[E_idx, t]) * R[E_idx, t])

        S_AMPA[:, t+1] = S_AMPA[:, t] + dS_AMPA * dt
        S_NMDA[:, t+1] = S_NMDA[:, t] + dS_NMDA * dt
        S_GABA[:, t+1] = S_GABA[:, t] + dS_GABA * dt

    # Boundary fill of last R column
    I_ext_last = np.zeros(4)
    if stim_on <= time[-1] <= stim_off:
        I_ext_last[0] = stim_strength
    I_tot_last = (I_back
                  + J_AMPA @ S_AMPA[:, -1]
                  + J_NMDA @ S_NMDA[:, -1]
                  + J_GABA @ S_GABA[:, -1]
                  + I_ext_last)
    R[E_idx, -1] = fI_excitatory(I_tot_last[E_idx])
    R[I_idx, -1] = fI_inhibitory(I_tot_last[I_idx])

    return {
        "time": time,
        "R": R,
        "S_AMPA": S_AMPA,
        "S_NMDA": S_NMDA,
        "S_GABA": S_GABA,
        "stim_on": stim_on,
        "stim_off": stim_off,
        "params": params,
    }


## Plotting

`plot_trial(result, ax)` draws one trial. `run_and_plot(params)` runs a **stimulus-strength sweep across the bifurcation point** and produces a single figure with one subplot per (stim, seed) trial. By default it uses `DEFAULT_TRIAL_SCHEDULE`, which spans clear miss → borderline → clear hit. Pass your own `trial_schedule` (a list of `(stim_nA, seed)` tuples) to override.


In [ ]:
_COLORS = {
    "Se":  "#56B4E9",   # sky blue
    "Si":  "#0072B2",   # darker blue (inhibitory of sensory)
    "FPe": "#E69F00",   # orange
    "FPi": "#D55E00",   # vermillion (inhibitory of FP)
}


def plot_trial(result, ax):
    """Plot firing rates of one CAI trial on the given axis."""
    t = result["time"]
    R = result["R"]
    ax.plot(t, R[0], color=_COLORS["Se"],  lw=2.5,
            label="Se (sensory E)", zorder=2)
    ax.plot(t, R[2], color=_COLORS["FPe"], lw=2.5,
            label="FPe (frontoparietal E)", zorder=3)
    # ax.plot(t, R[1], color=_COLORS["Si"],  lw=1.2, ls="--", alpha=0.85,
    #         label="Si (sensory I)", zorder=1)
    # ax.plot(t, R[3], color=_COLORS["FPi"], lw=1.2, ls="--", alpha=0.85,
    #         label="FPi (frontoparietal I)", zorder=1)
    ax.axvspan(result["stim_on"], result["stim_off"],
               color="grey", alpha=0.15, label="Stimulus", zorder=0)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Firing rate (Hz)")
    ax.grid(True, ls=":", alpha=0.6)


def _ignition_label(R, stim_off_idx):
    """Label whether ignition occurred based on FPe rate after stimulus."""
    # Look 200ms after stimulus offset for sustained activity
    fpe_late = R[2, stim_off_idx + 400:].mean()  # +200ms at dt=0.5ms
    return "HIT" if fpe_late > 5.0 else "MISS"


# Default trial schedule: a stimulus-strength sweep around the bifurcation.
# Each entry is (stim_strength_nA, seed). The bifurcation window for the
# current parameters is extremely narrow (~0.004 nA wide, centered at
# stim ~ 0.110 nA); at this value the network produces ~50% hits across
# seeds. Two trials at stim=0.110 with different seeds illustrate the
# noise-driven hit/miss stochasticity at threshold (Klatzmann 2025, Fig. 4B).
#
# If you change stim duration, stim_strength, or feedforward weights in
# PARAMS, re-run a stim sweep to find the new bifurcation point.
DEFAULT_TRIAL_SCHEDULE = [
    (0.050, 0),    # well below threshold -> clear miss
    (0.100, 0),    # approaching threshold -> miss with substantial transient
    (0.110, 7),    # at threshold, seed 7 -> miss (peak FPe ~20 Hz then decays)
    (0.110, 4),    # at threshold, seed 4 -> hit (same stim, different noise)
    (0.120, 0),    # just above threshold -> clean hit
    (0.200, 0),    # well above threshold -> strong transient, same attractor
]


def run_and_plot(params, trial_schedule=None, savedir=None, plot_ext=".png"):
    """Run a stimulus-strength sweep and plot one trial per stim value.

    Parameters
    ----------
    params : dict
        The PARAMS dict; its 'stim_strength' is overridden per trial.
    trial_schedule : list of (stim_strength, seed) tuples, or None
        If None, uses DEFAULT_TRIAL_SCHEDULE which sweeps from clear miss
        through the bifurcation to clear hit.
    savedir : str or None

    Returns None to prevent the Jupyter/Colab duplicate-figure issue.
    """
    if trial_schedule is None:
        trial_schedule = DEFAULT_TRIAL_SCHEDULE
    n_runs = len(trial_schedule)

    cols = min(n_runs, 3)
    rows = math.ceil(n_runs / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows),
                             sharex=True, sharey=True, squeeze=False)
    axes = axes.flatten()

    for i, (stim, seed) in enumerate(trial_schedule):
        # Override stim_strength for this trial only
        p_trial = {**params, "stim_strength": stim}
        result = run_simulation(p_trial, seed=seed)
        plot_trial(result, axes[i])
        stim_off_idx = int(result["stim_off"] / (result["time"][1] - result["time"][0]))
        label = _ignition_label(result["R"], stim_off_idx)
        axes[i].set_title(f"stim={stim:.3f} nA, seed={seed}: {label}", fontsize=11)

    for j in range(n_runs, len(axes)):
        axes[j].set_visible(False)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center right",
               bbox_to_anchor=(1.10, 0.5), fontsize=9)
    fig.suptitle(f"Conscious Access Ignition: stimulus-strength sweep across the bifurcation "
                 f"(noise={params['noise_type']})",
                 fontsize=13, y=1.02)

    plt.tight_layout()

    if savedir is not None:
        os.makedirs(savedir, exist_ok=True)
        fname = f"cai_sweep_{params['noise_type']}{plot_ext}"
        full = os.path.join(savedir, fname)
        plt.savefig(full, dpi=300, bbox_inches="tight")
        print(f"Saved: {full}")

    plt.show()
    plt.close(fig)
    return None


In [ ]:
PARAMS = {
    # --- Time grid (seconds) ---
    "time": {
        "trial_length": 3.0,
        "stim_on": 0.5,
        "stim_off": 0.55,
    },

    # --- External stimulus ---
    # Default value used when calling run_simulation() directly. The sweep in
    # run_and_plot() OVERRIDES this per-trial via its trial_schedule.
    # Sweep is done with the variable DEFAULT_TRIAL_SCHEDULE
    # "stim_strength": 1.0,         # nA, applied to Se only

    # --- Hierarchical AMPA/NMDA gradient (Klatzmann 2025 spine-count gradient) ---
    # Scales LOCAL recurrent self-excitation (NMDA + AMPA) per area.
    # Sensory areas are LESS excitable (lower h), FP MORE excitable.
    "hierarchy": {
        "h_sensory": 0.6,
        "h_frontoparietal": 1.0,
    },

    # --- Connectivity weights (nA) ---
    # All weights are stored as POSITIVE values; the sign for inhibitory
    # currents is applied when the matrix is assembled (I -> targets uses
    # a minus sign internally).
    "connectivity": {
        # Local recurrent self-excitation (E -> same E)
        # NOTE: scaled by h_sensory or h_frontoparietal at matrix assembly.
        "g_AMPA_self":      0.030,
        "g_NMDA_self":      0.310,    # tuned: 0.350 -> 0.310 brings FPe into 20-40 Hz target

        # Feedforward Se -> FPe (AMPA-dominant; Klatzmann)
        "g_AMPA_Se_to_FPe": 1.200, # 1.200, was 0.4

        # Feedback FPe -> Se and FPe -> Si (mixed AMPA + NMDA)
        "g_AMPA_FPe_to_Se": 0.010,
        "g_NMDA_FPe_to_Se": 0.010,
        "g_AMPA_FPe_to_Si": 0.030,
        "g_NMDA_FPe_to_Si": 0.150,    # tuned: 0.250 -> 0.150 reduces Si drive

        # Local E -> I (within each area)
        # NOTE: g_AMPA_FPe_to_FPi (0.85) is on the high end but consistent
        # with our WM notebook's g_AMPA_e_to_i (0.758). Both reflect that
        # AMPA gating S_AMPA = tau_AMPA * r is small (~0.06 at 30 Hz), so the
        # raw conductance must be sizeable to deliver meaningful current.
        # The 2.4x ratio over g_AMPA_Se_to_Si (0.35) mirrors the Klatzmann
        # AMPA/NMDA gradient (FP areas have higher AMPA contribution).
        "g_AMPA_Se_to_Si":   0.350,
        "g_AMPA_FPe_to_FPi": 0.850,   # tuned: 0.40 -> 0.85 brings FPi above FPe

        # Local I -> E (within each area; positive value, sign applied at assembly)
        "g_GABA_Si_to_Se":   0.250,
        "g_GABA_FPi_to_FPe": 0.300,

        # Local I -> I self-inhibition (positive value, sign applied at assembly)
        "g_GABA_Si":  0.250,
        "g_GABA_FPi": 0.150,
    },

    # --- Noise model (applied to all 4 populations) ---
    # "ou":       Ornstein-Uhlenbeck filtered by tau_AMPA (Klatzmann eq. 12, default)
    # "gaussian": iid Gaussian per step
    "noise_type": "ou",

    # --- Simulation control ---
    # 'seed' is used by run_simulation() when called directly. The sweep in
    # run_and_plot() uses per-trial seeds from its trial_schedule instead.
    "simulation": {
        "seed": 0,
    },
}


## Run

By default this runs the **bifurcation sweep**: 6 trials spanning stim=0.050 (clear miss) through stim=0.110 (borderline, two seeds chosen to straddle the threshold) to stim=0.200 (strong hit). The figure illustrates how a small change in stimulus strength flips the network from no ignition (miss) to sustained frontoparietal activity (hit), and how at threshold the same stimulus produces different outcomes depending on noise.

Key signatures of the bifurcation visible in the figure:
- **Panels 3 and 4 use the same `stim=0.110` with different seeds**: one misses, one hits. This is the noise-driven stochasticity at threshold.
- **All HIT trials converge to the same post-stimulus attractor** (FPe ~24 Hz) regardless of stim_strength — the hallmark of all-or-none ignition.
- The 50 ms stimulus duration (as in Klatzmann 2025) produces a crisp sensory transient that decays rapidly in miss trials, matching the biological constraint that unperceived stimuli should not leave sustained FP activity.

To run a custom schedule, pass `trial_schedule=[(stim, seed), ...]` to `run_and_plot()`.


In [ ]:
cai_savedir = os.path.join(saved_results, 'conscious_access_ignition')
run_and_plot(PARAMS, savedir=cai_savedir)


## Biological context

### What the model represents

The CAI model is a 2-area reduction of the Klatzmann et al. (2025) cortex-wide model of conscious access. It implements **Dehaene's global workspace theory** in mechanistic terms. The two areas stand for:

- **Sensory cortex** (Se, Si): early visual areas (V1, V2, MT). They process incoming stimuli but do not sustain activity in the absence of external drive.
- **Frontoparietal network** (FPe, FPi): a distributed network including dorsolateral PFC, posterior parietal cortex, and lateral prefrontal areas. This network can ignite into a sustained high-activity state that broadcasts the sensory signal globally — the proposed substrate of conscious access.

### The asymmetry between areas: hierarchical AMPA/NMDA gradient

Klatzmann's central anatomical claim is that **AMPA-to-NMDA receptor ratio decreases along the cortical hierarchy**, validated experimentally by in vitro autoradiography across 109 macaque cortical regions. In the model this means:

- **Sensory areas have low NMDA / high AMPA**: they respond fast (AMPA) but cannot sustain activity (no slow recurrent excitation). They are **monostable** — only a low-rate steady state.
- **Frontoparietal areas have high NMDA / low AMPA**: slow recurrent excitation supports persistent activity. They are **bistable** — both low-rate and high-rate steady states coexist.

Our `h_sensory` and `h_frontoparietal` parameters implement this gradient by scaling the local recurrent self-excitation. The sustained ~24 Hz FPe state after ignition is the model's high-activity attractor.

### The bifurcation mechanism for conscious access

Klatzmann (Fig. 4 of the paper) reframes conscious access as a **dynamic bifurcation**: a brief sensory cue temporarily reshapes the network's energy landscape, and depending on whether activity crosses a threshold, the network either falls back to baseline (a *miss*) or commits to the high-activity attractor (a *hit*).

In our 6-trial sweep:
- **stim ≤ 0.090 nA** (panels 1, 2): the cue is too weak to lift the network past the unstable separatrix. Activity returns to baseline → MISS.
- **stim ≈ 0.097 nA** (panels 3, 4): the cue lifts the network *near* the separatrix. Whether the trial commits to the high-activity attractor depends on whether the noise during the cue happened to push activity above or below the threshold. **Same stimulus, different outcomes** — exactly the "near-threshold variability" observed in psychophysical experiments on subliminal vs supraliminal perception.
- **stim ≥ 0.100 nA** (panels 5, 6): the cue is strong enough to cross the threshold reliably. Network ignites → HIT.

This narrow bifurcation window is the model's mechanistic explanation for why **subliminal stimuli produce no conscious report** despite some neural activity in sensory cortex: they don't push the FP system into the high-activity attractor.

### Feedforward AMPA, feedback mixed

Klatzmann's parameter search (Section "Fast propagation...") found that subnetwork bistability + realistic propagation times (130–200 ms) require:
- **Feedforward** Se→FPe pathway: AMPA-dominant (fast). This lets sensory information arrive at FP quickly.
- **Feedback** FPe→Se and FPe→Si pathways: mixed AMPA + NMDA, with greater NMDA contribution. This stabilizes ignition without producing runaway feedforward excitation.

In our model, `g_AMPA_Se_to_FPe = 0.40` is the fast feedforward channel; `g_NMDA_FPe_to_*` provides slow stabilizing feedback. Removing or weakening either channel destroys the ignition dynamics.

### What this model can and cannot tell you

It *can* tell you:
- Why ignition is all-or-none (it's a bifurcation, not a graded response)
- Why near-threshold stimuli produce mixed hit/miss outcomes (noise-driven crossings of the unstable separatrix)
- Why sensory and frontoparietal cortex show qualitatively different dynamics (the AMPA/NMDA gradient)
- Why feedforward and feedback streams have different receptor compositions

It *cannot* tell you:
- The cortex-wide propagation pattern (we have only 2 areas)
- Anything about the contents of consciousness or qualia (the model only addresses the access question)
- How attention selects which stimulus is reported when multiple are present (would require multiple Se modules)
- The role of subcortical structures (thalamus, basal ganglia) which Klatzmann does not model either
